# 🎮 AI Strategy Arena — Google Colab Edition
A polished interactive AI lab demonstrating Tic-Tac-Toe, Minimax, Alpha-Beta Pruning, live metrics, search traces, candidate move evaluations, and AI-vs-AI matches.

**Run the notebook from top to bottom.** The last cell creates a public Gradio link for the demo.

> Colab adaptation: this notebook uses Gradio instead of Streamlit because it launches interactively inside Colab with less setup. The algorithms are implemented from scratch.

In [ ]:
%pip -q install gradio plotly

In [ ]:
import math, time
from dataclasses import dataclass, field
import gradio as gr
import plotly.graph_objects as go

WINS=[(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]
EMPTY=[""]*9

def moves(b): return [i for i,x in enumerate(b) if not x]
def winner(b):
    for line in WINS:
        a,c,d=line
        if b[a] and b[a]==b[c]==b[d]: return b[a],line
    return None,None
def status(b):
    w,line=winner(b)
    if w: return f"{w} wins",line
    if not moves(b): return "Draw",None
    return "In progress",None
def next_player(b): return "X" if b.count("X")==b.count("O") else "O"
def valid_board(b):
    if len(b)!=9 or b.count("X") not in (b.count("O"),b.count("O")+1): return False
    x,_=winner(b); other=["O" if c=="X" else "X" if c=="O" else "" for c in b]; o,_=winner(other)
    return not (x and o) and not (x and b.count("X")!=b.count("O")+1) and not (o and b.count("X")!=b.count("O"))
def utility(b,ai,depth):
    w,_=winner(b)
    return 10-depth if w==ai else depth-10 if w else 0

@dataclass
class Stats:
    nodes:int=0
    leaves:int=0
    generated:int=0
    cutoffs:int=0
    max_depth:int=0
    trace:list=field(default_factory=list)

def ev(st,event,b,depth,role,val=None,a=None,be=None,move=None,note=""):
    # Keep the UI trace bounded: the algorithms still search the full tree,
    # but only the first 2,500 events are retained for visualization in Colab.
    if len(st.trace) < 2500:
        st.trace.append(dict(step=len(st.trace)+1,event=event,board=b.copy(),depth=depth,role=role,value=val,alpha=a,beta=be,move=move,note=note))

def minimax(b,depth,maximizing,ai,opp,st,parent=None):
    st.nodes+=1; st.max_depth=max(st.max_depth,depth); role="MAX" if maximizing else "MIN"
    ev(st,"enter",b,depth,role,move=parent,note=f"Enter {role} node.")
    s,_=status(b)
    if s!="In progress":
        st.leaves+=1; v=utility(b,ai,depth); ev(st,"terminal",b,depth,role,v,move=parent,note=f"{s}; utility {v}."); return v
    best=-math.inf if maximizing else math.inf
    for m in moves(b):
        child=b.copy(); child[m]=ai if maximizing else opp; st.generated+=1
        ev(st,"explore",child,depth+1,"MIN" if maximizing else "MAX",move=m,note=f"Explore cell {m+1}.")
        v=minimax(child,depth+1,not maximizing,ai,opp,st,m)
        best=max(best,v) if maximizing else min(best,v)
    ev(st,"return",b,depth,role,best,move=parent,note=f"{role} returns {best}.")
    return best

def alphabeta(b,depth,maximizing,ai,opp,st,a=-math.inf,be=math.inf,parent=None):
    st.nodes+=1; st.max_depth=max(st.max_depth,depth); role="MAX" if maximizing else "MIN"
    ev(st,"enter",b,depth,role,a=a,be=be,move=parent,note=f"Enter {role}; update bounds while searching.")
    s,_=status(b)
    if s!="In progress":
        st.leaves+=1; v=utility(b,ai,depth); ev(st,"terminal",b,depth,role,v,a,be,parent,f"{s}; utility {v}."); return v
    best=-math.inf if maximizing else math.inf; ms=moves(b)
    for j,m in enumerate(ms):
        child=b.copy(); child[m]=ai if maximizing else opp; st.generated+=1
        ev(st,"explore",child,depth+1,"MIN" if maximizing else "MAX",a=a,be=be,move=m,note=f"Explore cell {m+1}.")
        v=alphabeta(child,depth+1,not maximizing,ai,opp,st,a,be,m)
        best=max(best,v) if maximizing else min(best,v)
        if maximizing: a=max(a,best)
        else: be=min(be,best)
        ev(st,"bound",b,depth,role,best,a,be,m,f"Updated bounds: α={a}, β={be}.")
        if a>=be:
            st.cutoffs+=1
            ev(st,"cutoff",b,depth,role,best,a,be,m,f"α ≥ β, so skip {len(ms)-j-1} remaining sibling move(s).")
            break
    ev(st,"return",b,depth,role,best,a,be,parent,f"{role} returns {best}.")
    return best

def search(b,algo,ai):
    opp="O" if ai=="X" else "X"; st=Stats(); t=time.perf_counter(); vals=[]
    for m in moves(b):
        c=b.copy(); c[m]=ai; st.generated+=1
        v=minimax(c,1,False,ai,opp,st,m) if algo=="Minimax" else alphabeta(c,1,False,ai,opp,st,-math.inf,math.inf,m)
        vals.append((m,v))
    ms=(time.perf_counter()-t)*1000
    if not vals:return None,[],st,ms,None
    best=max(v for _,v in vals); return next(m for m,v in vals if v==best),vals,st,ms,best

def board_html(b,highlight=(),winline=()):
    out=[]
    for i,x in enumerate(b):
        cls="cell "+("x" if x=="X" else "o" if x=="O" else "")
        if i in highlight:cls+=" best"
        if i in winline:cls+=" win"
        out.append(f'<div class="{cls}">{x or i+1}</div>')
    return '<div class="board">'+"".join(out)+"</div>"

def metric_html(label,val,note):
    return f'<div class="metric"><div class="ml">{label}</div><div class="mv">{val}</div><div class="mn">{note}</div></div>'
def metrics(st,ms,val,move):
    return '<div class="metrics">'+''.join([
        metric_html("NODES ENTERED",f"{st.nodes:,}","recursive calls"),
        metric_html("TERMINAL LEAVES",f"{st.leaves:,}","finished states"),
        metric_html("PRUNING CUTOFFS",f"{st.cutoffs:,}","branches stopped early"),
        metric_html("MAX DEPTH",st.max_depth,"plies searched"),
        metric_html("ROOT VALUE","—" if val is None else f"{val:+.0f}","AI perspective"),
        metric_html("SELECTED CELL","—" if move is None else move+1,f"{ms:.3f} ms")
    ])+'</div>'
def bar_chart(vals,best=(),title="Candidate move evaluation"):
    fig=go.Figure(go.Bar(x=[f"Cell {m+1}" for m,v in vals],y=[v for m,v in vals],
        marker_color=["#36d9c5" if m in best else "#7985ff" for m,v in vals],
        text=[f"{v:+.0f}" for m,v in vals],textposition="outside"))
    fig.update_layout(title=title,height=300,margin=dict(l=15,r=15,t=50,b=20),
        paper_bgcolor="rgba(0,0,0,0)",plot_bgcolor="rgba(0,0,0,0)",font_color="#edf3ff",
        xaxis=dict(gridcolor="#283653"),yaxis=dict(title="Utility",gridcolor="#283653"))
    return fig
def tree_chart(st,limit=90):
    # Trace-derived layered node visualization; shows actual visited search states.
    entries=[e for e in st.trace if e["event"] in ("enter","terminal","cutoff","return")]
    entries=entries[:limit]
    if not entries:
        fig=go.Figure();fig.update_layout(title="Run a search to reveal the tree",paper_bgcolor="rgba(0,0,0,0)",plot_bgcolor="rgba(0,0,0,0)",font_color="#edf3ff");return fig
    layers={}
    for e in entries:layers.setdefault(e["depth"],[]).append(e)
    xs=[];ys=[];texts=[];colors=[]
    for d,es in layers.items():
        for j,e in enumerate(es):
            xs.append((j+1)/(len(es)+1));ys.append(-d)
            texts.append(f'{e["role"]}<br>d={d}<br>{ "".join(x or "·" for x in e["board"])}<br>{e["event"]}')
            colors.append("#f4bd58" if e["event"]=="cutoff" else "#37d9c5" if e["role"]=="MAX" else "#818cff")
    fig=go.Figure(go.Scatter(x=xs,y=ys,mode="markers+text",text=texts,textposition="bottom center",
        marker=dict(size=16,color=colors,line=dict(color="#dfe8ff",width=1)),hovertemplate="%{text}<extra></extra>"))
    fig.update_layout(title=f"Search trace map · first {len(entries)} recorded states",height=430,
        paper_bgcolor="rgba(0,0,0,0)",plot_bgcolor="rgba(0,0,0,0)",font_color="#edf3ff",
        xaxis=dict(visible=False),yaxis=dict(title="Depth",gridcolor="#283653"),margin=dict(l=10,r=10,t=50,b=15))
    return fig
def trace_html(st,start=0,count=18):
    if not st.trace:return '<div class="empty">Run a search to generate a real execution trace.</div>'
    out=[]
    for e in st.trace[start:start+count]:
        cls="trace"; cls+= " cut" if e["event"]=="cutoff" else " term" if e["event"]=="terminal" else ""
        val="—" if e["value"] is None else f'{e["value"]:+.0f}'
        bounds=""
        if e["alpha"] is not None or e["beta"] is not None:
            def fmt(x): return "−∞" if x==-math.inf else "∞" if x==math.inf else str(round(x,2))
            bounds=f'<div class="bound">α {fmt(e["alpha"])} · β {fmt(e["beta"])}</div>'
        out.append(f'<div class="{cls}"><div class="num">{e["step"]:02}</div><div><b>{e["event"].upper()}</b> <span class="role">{e["role"]} · depth {e["depth"]}</span><span class="value">v={val}</span><div class="note">{e["note"]}</div>{bounds}</div></div>')
    return '<div class="tracebox">'+"".join(out)+"</div>"

CSS="""
.gradio-container{max-width:1450px!important;background:radial-gradient(ellipse at 12% 0%,#1a2850 0%,#0a1020 43%,#070b14 100%)!important;color:#edf3ff!important}
#hero{padding:28px;border:1px solid #2b3b60;border-radius:23px;background:linear-gradient(120deg,#1d315d99,#17203a99,#17434a55);margin-bottom:16px}
.eyebrow{font-size:11px;letter-spacing:.18em;font-weight:850;color:#49dfcc}.hero-title{font-size:clamp(32px,4vw,48px);font-weight:900;margin:9px 0}.hero-sub{color:#b9c7e2;line-height:1.7}.pills{display:flex;gap:8px;flex-wrap:wrap;margin-top:14px}.pill{padding:6px 10px;border:1px solid #34496c;border-radius:30px;background:#14223d;color:#dbe8ff;font-size:11px}
.board{display:grid;grid-template-columns:repeat(3,minmax(65px,105px));gap:9px;justify-content:center;padding:14px}.cell{height:92px;display:flex;align-items:center;justify-content:center;border:1px solid #344665;border-radius:16px;background:linear-gradient(145deg,#192541,#11192b);font-size:35px;font-weight:900;color:#61718f}.cell.x{color:#48dfcd;text-shadow:0 0 18px #48dfcd44}.cell.o{color:#a8a1ff;text-shadow:0 0 18px #a8a1ff44}.cell.best{border-color:#38d8c5;box-shadow:0 0 0 2px #38d8c533,0 0 25px #38d8c51a}.cell.win{background:#17443d;border-color:#38d8c5}
.metrics{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:9px}.metric{padding:13px;border:1px solid #293957;border-radius:15px;background:linear-gradient(150deg,#17223a,#10182a);min-height:94px}.ml{font-size:10px;color:#9fb0d1;letter-spacing:.1em;font-weight:800}.mv{font-size:25px;font-weight:900;margin-top:5px}.mn{font-size:10px;color:#94a3c2}
.tracebox{display:flex;flex-direction:column;gap:8px;max-height:570px;overflow-y:auto}.trace{display:flex;gap:10px;padding:12px;border:1px solid #293956;border-radius:12px;background:#121c30}.trace.cut{border-color:#a67c35;background:#3a2b183d}.trace.term{border-color:#2b8f80}.num{width:30px;height:30px;display:flex;align-items:center;justify-content:center;background:#243451;border-radius:8px;color:#55e3d2;font-weight:850;font-size:11px;flex-shrink:0}.role{font-size:11px;color:#b4c2de;background:#25324c;border-radius:20px;padding:3px 7px}.value{float:right;color:#57e4d1;font-weight:800}.note{font-size:12px;color:#c2cee4;margin-top:7px;line-height:1.5}.bound{font-size:11px;color:#f4c36c;margin-top:6px}.empty{padding:24px;text-align:center;border:1px dashed #3b4c6e;border-radius:14px;color:#9fb0ce}.small{color:#9eacc8;font-size:12px;line-height:1.6}.section{padding:16px;border:1px solid #2b3956;border-radius:16px;background:#111a2b99}
button.primary{background:linear-gradient(100deg,#20b9ac,#526ff4)!important;color:white!important;font-weight:800!important}
"""

HERO="""<div id="hero"><div class="eyebrow">CLASSICAL AI · INTERACTIVE LAB</div><div class="hero-title">AI Strategy Arena</div><div class="hero-sub">Explore how artificial intelligence thinks. Play Tic-Tac-Toe against an optimal agent, inspect its search process, and compare Minimax with Alpha-Beta Pruning using live metrics.</div><div class="pills"><span class="pill">MINIMAX</span><span class="pill">ALPHA-BETA PRUNING</span><span class="pill">SEARCH VISUALIZATION</span><span class="pill">LIVE METRICS</span></div></div>"""

EXAMPLES={
"Empty board":[""]*9,
"Immediate win available":["X","X","","O","O","","","",""],
"Must block opponent":["O","O","","X","","","","X",""],
"Multiple optimal moves":["X","","","","O","","","","X"],
"Near draw":["X","O","X","X","O","","O","X",""]
}

def run_analysis(scenario,custom,algo,ai,start):
    if scenario=="Custom board (9 chars)":
        raw=(custom or "").upper().replace(" ","").replace(",","").replace("|","").replace(".","-").replace("_","-")
        if len(raw)!=9 or any(c not in "XO-" for c in raw):
            st=Stats();return "⚠️ Enter exactly 9 characters using X, O, and - for empty cells.",board_html(EMPTY),metrics(st,0,None,None),bar_chart([]),tree_chart(st),trace_html(st)
        b=["" if c=="-" else c for c in raw]
    else:b=EXAMPLES[scenario].copy()
    if not valid_board(b) or status(b)[0]!="In progress":
        st=Stats();return "⚠️ This is not a valid non-terminal position.",board_html(b),metrics(st,0,None,None),bar_chart([]),tree_chart(st),trace_html(st)
    move,vals,st,ms,v=search(b,algo,ai)
    # Exact candidate root values for highlighting equally optimal moves.
    opp="O" if ai=="X" else "X"; exact=[]
    for m in moves(b):
        c=b.copy();c[m]=ai;tmp=Stats()
        vv=minimax(c,1,False,ai,opp,tmp,m) if algo=="Minimax" else alphabeta(c,1,False,ai,opp,tmp,-math.inf,math.inf,m)
        exact.append((m,vv))
    best=max(x[1] for x in exact); bm=[m for m,x in exact if x==best]
    return f"**Analysis complete.** {algo} recommends cell {bm[0]+1}; best utility {best:+.0f}. Equally optimal cells: {', '.join(str(x+1) for x in bm)}.",board_html(b,bm),metrics(st,ms,best,bm[0]),bar_chart(exact,bm),tree_chart(st),trace_html(st,int(start or 0))

def showdown(scenario,ai):
    b=EXAMPLES[scenario].copy();opp="O" if ai=="X" else "X"
    m,vm,sm,tmm,vmm=search(b,"Minimax",ai);a,va,sa,tma,vma=search(b,"Alpha-Beta Pruning",ai)
    # Exact best move sets from the candidate scores returned at root.
    bm=max(v for _,v in vm);ba=max(v for _,v in va)
    setm={x for x,v in vm if v==bm};seta={x for x,v in va if v==ba}
    same=bm==ba and setm==seta
    reduction=(sm.nodes-sa.nodes)/sm.nodes*100 if sm.nodes else 0
    summary=f"""<div class="section"><div class="eyebrow">{'✓ SAME OPTIMAL RESULT' if same else 'CHECK RESULTS'}</div><h3>Same board · same utility · same move order</h3><p>Root value: Minimax <b>{bm:+.0f}</b> · Alpha-Beta <b>{ba:+.0f}</b></p><p>Optimal cells: Minimax <b>{', '.join(str(x+1) for x in sorted(setm))}</b> · Alpha-Beta <b>{', '.join(str(x+1) for x in sorted(seta))}</b></p><p>Nodes entered: <b>{sm.nodes:,}</b> vs <b>{sa.nodes:,}</b> · Node-count reduction <b>{reduction:.1f}%</b></p><p>Alpha-Beta cutoffs: <b>{sa.cutoffs}</b> · Runtime: {tmm:.3f} ms vs {tma:.3f} ms</p><div class="small">These are measurements from this run. Runtime may vary; node reduction is not an energy-saving measurement.</div></div>"""
    fig=go.Figure()
    fig.add_trace(go.Bar(name="Nodes entered",x=["Minimax","Alpha-Beta"],y=[sm.nodes,sa.nodes],marker_color=["#818cff","#36d9c5"]))
    fig.add_trace(go.Bar(name="Terminal leaves",x=["Minimax","Alpha-Beta"],y=[sm.leaves,sa.leaves],marker_color=["#b28cff","#f5bd58"]))
    fig.update_layout(barmode="group",title="Actual search work",height=330,paper_bgcolor="rgba(0,0,0,0)",plot_bgcolor="rgba(0,0,0,0)",font_color="#edf3ff",yaxis=dict(gridcolor="#283653"))
    return ("**Both algorithms agree on the optimal result.**" if same else "**Result mismatch: inspect implementation.**"),board_html(b,setm),fig,summary,bar_chart(vm,setm,"Minimax root candidate values")

def play(cell,human,algo,state):
    state=state or {"board":EMPTY.copy(),"history":[]}
    b=state["board"].copy();idx=int(cell)-1
    if status(b)[0]!="In progress":return state,"Game over — start a new game.",board_html(b),metrics(Stats(),0,None,None),bar_chart([]),""
    if next_player(b)!=human:return state,"It is not your turn. The AI moves first when you choose O.",board_html(b),metrics(Stats(),0,None,None),bar_chart([]),""
    if idx not in moves(b):return state,"That cell is occupied. Choose an empty cell.",board_html(b),metrics(Stats(),0,None,None),bar_chart([]),""
    state["history"].append(b.copy());b[idx]=human;ai="O" if human=="X" else "X"
    st=Stats();vals=[];ms=0;v=None;move=None
    if status(b)[0]=="In progress":
        move,vals,st,ms,v=search(b,algo,ai)
        if move is not None:b[move]=ai
    s,line=status(b);state.update(board=b,human=human,algo=algo,last_stats=st,last_vals=vals,last_ms=ms,last_v=v,last_move=move)
    msg=f"**{s}.** You played cell {idx+1}."
    if move is not None:msg+=f" {algo} chose cell {move+1} with root utility {v:+.0f}."
    return state,msg,board_html(b,[move] if move is not None else [],line or ()),metrics(st,ms,v,move),bar_chart(vals,[move] if move is not None else []),trace_html(st)

def reset(human,algo):
    st={"board":EMPTY.copy(),"history":[],"human":human,"algo":algo,"last_stats":Stats(),"last_vals":[],"last_ms":0,"last_v":None,"last_move":None}
    return st,"New game started. Choose a cell to make your move.",board_html(EMPTY),metrics(Stats(),0,None,None),bar_chart([]),trace_html(Stats())
def undo(state):
    state=state or {"board":EMPTY.copy(),"history":[]}
    if not state.get("history"):return state,"No move to undo.",board_html(state["board"]),metrics(Stats(),0,None,None),bar_chart([]),trace_html(Stats())
    state["board"]=state["history"].pop();st=Stats()
    return state,"Move undone.",board_html(state["board"]),metrics(st,0,None,None),bar_chart([]),trace_html(st)

def aivai(scenario,xalgo,oalgo,plies):
    b=EXAMPLES[scenario].copy();logs=[]
    for ply in range(int(plies)):
        if status(b)[0]!="In progress":break
        p=next_player(b);algo=xalgo if p=="X" else oalgo
        m,vals,st,ms,v=search(b,algo,p)
        if m is None:break
        b[m]=p;logs.append(f"| {ply+1} | {p} | {algo} | {m+1} | {v:+.0f} | {st.nodes} | {st.cutoffs} |")
    s,line=status(b)
    table="| Turn | Player | Algorithm | Cell | Value | Nodes | Cutoffs |\\n|---:|---|---|---:|---:|---:|---:|\\n"+"\\n".join(logs)
    return f"**Result:** {s}. X uses {xalgo}; O uses {oalgo}.",board_html(b,(),line or ()),table

LEARN="""<div class="section"><h2>How the AI decides</h2><p><b>1 · State:</b> the board is the current state; each empty cell is a legal action.</p><p><b>2 · Game tree:</b> each edge is a move. MAX is the AI; MIN is its opponent.</p><p><b>3 · Utility:</b> AI win = +10 adjusted by depth, loss = −10 adjusted by depth, draw = 0.</p><p><b>4 · Backtracking:</b> terminal scores propagate upward. MAX chooses the highest child value; MIN chooses the lowest.</p><p><b>5 · Pruning:</b> α is MAX's best bound; β is MIN's best bound. When α ≥ β, remaining siblings cannot change the decision and are skipped.</p></div><div class="section" style="margin-top:12px"><h2>Complexity</h2><p>Minimax worst case: O(b<sup>d</sup>). Alpha-Beta worst case is also O(b<sup>d</sup>), and with ideal move ordering can approach O(b<sup>d/2</sup>).</p><p class="small">b is branching factor and d is depth. These are theoretical bounds, not measured app results.</p><h2>Lab alignment</h2><p>Experiment 6: Minimax for Tic-Tac-Toe. Experiment 7: Alpha-Beta Pruning.</p><p>This is classical adversarial search, not machine learning. The agent does not train or learn from past games.</p></div>"""

with gr.Blocks(css=CSS,theme=gr.themes.Soft(primary_hue="teal",secondary_hue="indigo",neutral_hue="slate"),title="AI Strategy Arena") as app:
    gr.HTML(HERO)
    with gr.Tabs():
        with gr.Tab("🎮 Play Arena"):
            state=gr.State({"board":EMPTY.copy(),"history":[],"human":"X","algo":"Alpha-Beta Pruning","last_stats":Stats(),"last_vals":[],"last_ms":0,"last_v":None,"last_move":None})
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("## Play against AI")
                    human=gr.Radio(["X","O"],value="X",label="You play as")
                    algo=gr.Radio(["Minimax","Alpha-Beta Pruning"],value="Alpha-Beta Pruning",label="AI algorithm")
                    board=gr.HTML(board_html(EMPTY))
                    with gr.Row():buttons=[gr.Button(f"Cell {i+1}") for i in range(9)]
                    with gr.Row():
                        new=gr.Button("↻ New game",variant="primary");undo_btn=gr.Button("↶ Undo")
                    play_msg=gr.Markdown("Ready. Select a numbered cell below the board.")
                with gr.Column(scale=1):
                    gr.Markdown("## Live AI telemetry")
                    play_metrics=gr.HTML(metrics(Stats(),0,None,None))
                    play_plot=gr.Plot(value=bar_chart([]))
                    with gr.Accordion("Latest search trace",open=False):play_trace=gr.HTML(trace_html(Stats()))
            outs=[state,play_msg,board,play_metrics,play_plot,play_trace]
            for i,btn in enumerate(buttons):
                btn.click(lambda h,a,s,n=i+1:play(n,h,a,s),[human,algo,state],outs)
            new.click(reset,[human,algo],outs)
            undo_btn.click(undo,[state],outs)
        with gr.Tab("🧠 AI Thinking Lab"):
            gr.Markdown("## Inspect the search\nChoose a position and algorithm. The trace and tree are generated by the actual recursive search.")
            with gr.Row():
                with gr.Column(scale=1):
                    scenario=gr.Dropdown(list(EXAMPLES.keys())+["Custom board (9 chars)"],value="Immediate win available",label="Board position")
                    custom=gr.Textbox(value="XX-OO----",label="Custom board (X/O/-)")
                    lab_algo=gr.Radio(["Minimax","Alpha-Beta Pruning"],value="Alpha-Beta Pruning",label="Algorithm")
                    ai=gr.Radio(["X","O"],value="X",label="MAX / AI symbol")
                    start=gr.Slider(0,100,value=0,step=1,label="Trace start event")
                    run=gr.Button("▶ Run analysis",variant="primary")
                    lab_msg=gr.Markdown()
                    lab_board=gr.HTML(board_html(EMPTY))
                    lab_metrics=gr.HTML(metrics(Stats(),0,None,None))
                with gr.Column(scale=2):
                    lab_plot=gr.Plot(value=bar_chart([]));lab_tree=gr.Plot(value=tree_chart(Stats()))
            with gr.Accordion("Actual search trace",open=True):lab_trace=gr.HTML(trace_html(Stats()))
            run.click(run_analysis,[scenario,custom,lab_algo,ai,start],[lab_msg,lab_board,lab_metrics,lab_plot,lab_tree,lab_trace])
            start.change(run_analysis,[scenario,custom,lab_algo,ai,start],[lab_msg,lab_board,lab_metrics,lab_plot,lab_tree,lab_trace])
        with gr.Tab("⚔️ Algorithm Showdown"):
            gr.Markdown("## Same position. Same utility. Compare actual search work.")
            with gr.Row():
                with gr.Column(scale=1):
                    sh_scenario=gr.Dropdown(list(EXAMPLES.keys()),value="Empty board",label="Benchmark position")
                    sh_ai=gr.Radio(["X","O"],value="X",label="AI / MAX symbol")
                    sh_run=gr.Button("⚡ Compare algorithms",variant="primary")
                    sh_msg=gr.Markdown();sh_board=gr.HTML(board_html(EMPTY));sh_summary=gr.HTML('<div class="empty">Run a comparison.</div>')
                with gr.Column(scale=2):
                    sh_bar=gr.Plot(value=go.Figure());sh_moves=gr.Plot(value=bar_chart([]))
            sh_run.click(showdown,[sh_scenario,sh_ai],[sh_msg,sh_board,sh_bar,sh_summary,sh_moves])
        with gr.Tab("🤖 AI vs AI"):
            gr.Markdown("## Watch the algorithms play a complete match")
            with gr.Row():
                with gr.Column(scale=1):
                    av_scenario=gr.Dropdown(list(EXAMPLES.keys()),value="Empty board",label="Starting position")
                    xalgo=gr.Radio(["Minimax","Alpha-Beta Pruning"],value="Minimax",label="Player X")
                    oalgo=gr.Radio(["Minimax","Alpha-Beta Pruning"],value="Alpha-Beta Pruning",label="Player O")
                    plies=gr.Slider(1,9,value=9,step=1,label="Maximum additional moves")
                    av_run=gr.Button("▶ Run match",variant="primary")
                    av_msg=gr.Markdown();av_board=gr.HTML(board_html(EMPTY))
                with gr.Column(scale=2):av_log=gr.Markdown("Move-by-move telemetry appears here.")
            av_run.click(aivai,[av_scenario,xalgo,oalgo,plies],[av_msg,av_board,av_log])
        with gr.Tab("📚 Learn AI"):
            gr.HTML(LEARN)
            gr.Markdown("### Viva explanation\n1. Represent the board as a state and empty cells as actions.\n2. Search the future game tree until terminal states.\n3. Assign utilities and propagate values back through recursion.\n4. MAX chooses the largest value; MIN chooses the smallest.\n5. Alpha-Beta uses bounds to prune branches that cannot change the root decision.\n6. Compare both algorithms on the same board and verify the optimal value and move set.")
    gr.Markdown("<div class='small' style='text-align:center;padding:18px'>AI Strategy Arena · Classical adversarial search · Built for an AI lab demonstration</div>")

app.queue().launch(share=True, debug=False)